# H3SFCA 문화누리 접근성 분석

- 목적: 문화누리 가맹점 접근성 테이블을 이용해 격자별·중분류별 H3SFCA 접근성 지수를 산출함.
- 설계: 접근 가능한 가맹점 후보 안에서 Gaussian 거리감쇠만으로 Huff 배분확률을 계산함.
- 산출물: 격자-중분류 H3SFCA 접근성, 가맹점별 공급-수요비, 격자 요약 테이블을 생성함.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["font.family"] = "Noto Sans KR"
plt.rcParams["axes.unicode_minus"] = False

BASE_PATH = Path().resolve()

if BASE_PATH.name == "access":
    BASE_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "oracle_mnc_project").exists():
    BASE_PATH = BASE_PATH / "oracle_mnc_project"

ANALYSIS_OUTPUT_PATH = BASE_PATH / "analysis_table" / "data" / "output"
ACCESS_OUTPUT_PATH = BASE_PATH / "notebooks" / "access" / "OUTPUT" / "h3sfca"
ACCESS_IMAGE_PATH = BASE_PATH / "notebooks" / "access" / "IMAGE" / "h3sfca"

ACCESS_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
ACCESS_IMAGE_PATH.mkdir(parents=True, exist_ok=True)

ACCESS_TABLE_PATH = ANALYSIS_OUTPUT_PATH / "문화누리_격자_가맹점_접근성_통합테이블.csv"

print("BASE_PATH:", BASE_PATH)
print("접근성 테이블 존재:", ACCESS_TABLE_PATH.exists())
print("접근성 테이블 용량(MB):", round(ACCESS_TABLE_PATH.stat().st_size / 1024 / 1024, 2))

## 사용 데이터 점검

- 기존 도보·대중교통 네트워크 분석 결과를 결합한 `문화누리_격자_가맹점_접근성_통합테이블.csv`를 사용함.
- 접근비용은 도보의 경우 m, 대중교통의 경우 분 단위로 해석함.
- 문화누리 접근성 분석 대상 중분류만 사용하고, 교통수단·숙박·여행사 등 접근성 분석 제외 분류는 포함하지 않음.

In [ ]:
sample = pd.read_csv(ACCESS_TABLE_PATH, nrows=5, low_memory=False)

print("샘플 구조:", sample.shape)
print("칼럼 목록")
print(sample.columns.tolist())
display(sample.head())

required_cols = [
    "GRID_CD", "가맹점_ID", "접근수단", "접근비용",
    "시군구_격자", "행정동_격자", "문화누리대상자_추정_인구수",
    "가맹점명", "시군구_가맹점", "대분류", "중분류", "소분류"
]

missing_cols = [col for col in required_cols if col not in sample.columns]
print("필수 칼럼 누락:", missing_cols)

if missing_cols:
    raise ValueError(f"필수 칼럼이 없습니다: {missing_cols}")

## H3SFCA 산식

- Gaussian 거리감쇠

\[
G(d_{ij}) = \exp\left(-\frac{1}{2}\left(\frac{d_{ij}}{\sigma}\right)^2\right)
\]

- Huff 배분확률

\[
H_{ijc} = \frac{G(d_{ij})}{\sum_{k \in J_{ic}}G(d_{ik})}
\]

- 가맹점별 유효수요

\[
D_{jc} = \sum_i D_{ic}H_{ijc}
\]

- 가맹점별 공급-수요비

\[
R_{jc} = \frac{S_{jc}}{D_{jc}}
\]

- 격자별 H3SFCA 접근성

\[
A_{ic} = \sum_j H_{ijc}R_{jc}
\]

- 본 노트북에서는 Huff 배분확률 계산 시 공급량을 매력도로 넣지 않고, 거리감쇠만 사용함.
- 머신러닝 기반 중분류별 선호확률 테이블이 아직 결합되지 않았으므로, 수요량 \(D_{ic}\)은 현재 단계에서 격자별 문화누리대상자 추정 인구수로 설정함.

In [ ]:
category_list = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품", "공연", "관광지", "미술", "스포츠관람"]
mode_cutoff = {
    "도보": 750,      # m
    "대중교통": 20    # min
}

cutoff_weight = 0.05
chunk_size = 500_000

usecols = [
    "GRID_CD", "가맹점_ID", "접근수단", "접근비용",
    "시군구_격자", "행정동_격자", "문화누리대상자_추정_인구수",
    "가맹점명", "시군구_가맹점", "대분류", "중분류", "소분류"
]

def gaussian_sigma(cutoff, target_weight=0.05):
    return cutoff / np.sqrt(-2 * np.log(target_weight))

def add_gaussian_weight(df):
    df = df.copy()
    df["접근비용"] = pd.to_numeric(df["접근비용"], errors="coerce")
    df["문화누리대상자_추정_인구수"] = pd.to_numeric(
        df["문화누리대상자_추정_인구수"], errors="coerce"
    ).fillna(0)
    df = df[df["접근수단"].isin(mode_cutoff.keys())].copy()
    df = df[df["중분류"].isin(category_list)].copy()
    df = df[df["접근비용"].notna() & (df["접근비용"] >= 0)].copy()
    df["cutoff"] = df["접근수단"].map(mode_cutoff)
    df = df[df["접근비용"] <= df["cutoff"]].copy()
    df["sigma"] = df["cutoff"].map(lambda x: gaussian_sigma(x, cutoff_weight))
    df["거리감쇠"] = np.exp(-0.5 * (df["접근비용"] / df["sigma"]) ** 2)
    return df

def add_series(acc, part):
    if acc is None:
        return part
    return acc.add(part, fill_value=0)

print("분석 중분류:", category_list)
print("접근수단별 cutoff:", mode_cutoff)
print("cutoff 지점 감쇠값:", cutoff_weight)

## 1. 거리감쇠 합계 생성

- 접근 가능한 격자-가맹점 pair별로 Gaussian 거리감쇠값을 계산함.
- 같은 격자·접근수단·중분류 안에서 거리감쇠 합계를 생성하여 Huff 배분확률의 분모로 사용함.
- 격자 수요 정보와 가맹점 기본 정보는 이후 단계에서 재사용하기 위해 함께 정리함.

In [ ]:
sum_weight_series = None
row_count_series = None
grid_meta_list = []
store_meta_list = []
mode_category_list = []
processed_rows = 0
valid_rows = 0

for n, chunk in enumerate(pd.read_csv(
    ACCESS_TABLE_PATH,
    usecols=usecols,
    chunksize=chunk_size,
    low_memory=False
), start=1):
    processed_rows += len(chunk)
    chunk = add_gaussian_weight(chunk)
    valid_rows += len(chunk)

    group_cols = ["접근수단", "GRID_CD", "중분류"]
    sum_part = chunk.groupby(group_cols)["거리감쇠"].sum()
    count_part = chunk.groupby(group_cols)["거리감쇠"].size()

    sum_weight_series = add_series(sum_weight_series, sum_part)
    row_count_series = add_series(row_count_series, count_part)

    grid_meta_list.append(
        chunk[["GRID_CD", "시군구_격자", "행정동_격자", "문화누리대상자_추정_인구수"]]
        .drop_duplicates("GRID_CD")
    )
    store_meta_list.append(
        chunk[["가맹점_ID", "가맹점명", "시군구_가맹점", "대분류", "중분류", "소분류"]]
        .drop_duplicates(["가맹점_ID", "중분류"])
    )
    mode_category_list.append(
        chunk.groupby(["접근수단", "중분류"], as_index=False).size()
    )

    if n % 5 == 0:
        print(f"{n:,}개 chunk 처리 / 누적 유효 row: {valid_rows:,}")

sum_weight = sum_weight_series.reset_index(name="거리감쇠합")
row_count = row_count_series.reset_index(name="접근가능_가맹점수")

grid_meta = (
    pd.concat(grid_meta_list, ignore_index=True)
    .sort_values(["GRID_CD", "문화누리대상자_추정_인구수"], ascending=[True, False])
    .drop_duplicates("GRID_CD")
)

store_meta = (
    pd.concat(store_meta_list, ignore_index=True)
    .drop_duplicates(["가맹점_ID", "중분류"])
)

mode_category_count = (
    pd.concat(mode_category_list, ignore_index=True)
    .groupby(["접근수단", "중분류"], as_index=False)["size"]
    .sum()
    .rename(columns={"size": "pair_row수"})
)

print("전체 입력 row:", f"{processed_rows:,}")
print("분석 사용 row:", f"{valid_rows:,}")
print("거리감쇠합 테이블:", sum_weight.shape)
print("격자 meta:", grid_meta.shape)
print("가맹점 meta:", store_meta.shape)
display(mode_category_count)

## 2. 시설별 유효수요 계산

- 각 격자 수요를 같은 중분류 내 접근 가능한 가맹점으로 Huff 배분확률에 따라 나눔.
- 현재 수요량은 격자별 문화누리대상자 추정 인구수를 사용함.
- 거리감쇠합이 0인 경우는 배분확률을 계산할 수 없으므로 제외함.

In [ ]:
sum_weight_for_merge = sum_weight.copy()
grid_demand = grid_meta[["GRID_CD", "문화누리대상자_추정_인구수"]].copy()
grid_demand = grid_demand.rename(columns={"문화누리대상자_추정_인구수": "수요인구수"})

facility_demand_series = None
allocated_rows = 0

for n, chunk in enumerate(pd.read_csv(
    ACCESS_TABLE_PATH,
    usecols=usecols,
    chunksize=chunk_size,
    low_memory=False
), start=1):
    chunk = add_gaussian_weight(chunk)
    chunk = chunk.merge(
        sum_weight_for_merge,
        on=["접근수단", "GRID_CD", "중분류"],
        how="left"
    )
    chunk = chunk.merge(grid_demand, on="GRID_CD", how="left")
    chunk = chunk[(chunk["거리감쇠합"] > 0) & chunk["수요인구수"].notna()].copy()

    chunk["Huff_배분확률"] = chunk["거리감쇠"] / chunk["거리감쇠합"]
    chunk["배분수요"] = chunk["수요인구수"] * chunk["Huff_배분확률"]

    part = chunk.groupby(["접근수단", "가맹점_ID", "중분류"])["배분수요"].sum()
    facility_demand_series = add_series(facility_demand_series, part)
    allocated_rows += len(chunk)

    if n % 5 == 0:
        print(f"{n:,}개 chunk 처리 / 누적 배분 row: {allocated_rows:,}")

facility_demand = facility_demand_series.reset_index(name="시설_유효수요")

print("배분에 사용된 row:", f"{allocated_rows:,}")
print("시설별 유효수요 테이블:", facility_demand.shape)
display(facility_demand.head())

## 3. 공급-수요비 계산

- 가맹점별 공급량은 기본값 1로 설정함.
- 본 단계의 공급량은 시설 규모 차등 공급량이 아니라, 가맹점 1개소를 동일 공급 단위로 보는 기준값임.
- 시설 유효수요가 0인 경우 공급-수요비는 계산 안정성을 위해 0으로 처리함.

In [ ]:
store_supply = store_meta[["가맹점_ID", "중분류", "가맹점명", "시군구_가맹점", "소분류"]].copy()
store_supply["공급량"] = 1.0

facility_ratio = facility_demand.merge(
    store_supply,
    on=["가맹점_ID", "중분류"],
    how="left"
)

facility_ratio["공급량"] = facility_ratio["공급량"].fillna(1.0)
facility_ratio["시설_공급수요비"] = np.where(
    facility_ratio["시설_유효수요"] > 0,
    facility_ratio["공급량"] / facility_ratio["시설_유효수요"],
    0
)

print("시설 공급-수요비 테이블:", facility_ratio.shape)
print("시설 유효수요 결측:", facility_ratio["시설_유효수요"].isna().sum())
print("공급-수요비 요약")
display(facility_ratio["시설_공급수요비"].describe())
display(facility_ratio.head())

## 4. 격자별 H3SFCA 접근성 계산

- 격자에서 접근 가능한 가맹점의 공급-수요비를 Huff 배분확률로 가중합함.
- 결과는 격자·접근수단·중분류 단위의 H3SFCA 접근성 지수임.
- 값이 클수록 해당 격자에서 해당 중분류 시설의 수요 대비 공급 접근성이 높다는 뜻임.

In [ ]:
ratio_for_merge = facility_ratio[["접근수단", "가맹점_ID", "중분류", "시설_공급수요비"]].copy()
grid_access_series = None
access_rows = 0

for n, chunk in enumerate(pd.read_csv(
    ACCESS_TABLE_PATH,
    usecols=usecols,
    chunksize=chunk_size,
    low_memory=False
), start=1):
    chunk = add_gaussian_weight(chunk)
    chunk = chunk.merge(
        sum_weight_for_merge,
        on=["접근수단", "GRID_CD", "중분류"],
        how="left"
    )
    chunk = chunk.merge(
        ratio_for_merge,
        on=["접근수단", "가맹점_ID", "중분류"],
        how="left"
    )
    chunk = chunk[(chunk["거리감쇠합"] > 0) & chunk["시설_공급수요비"].notna()].copy()

    chunk["Huff_배분확률"] = chunk["거리감쇠"] / chunk["거리감쇠합"]
    chunk["H3SFCA_기여값"] = chunk["Huff_배분확률"] * chunk["시설_공급수요비"]

    part = chunk.groupby(["접근수단", "GRID_CD", "중분류"])["H3SFCA_기여값"].sum()
    grid_access_series = add_series(grid_access_series, part)
    access_rows += len(chunk)

    if n % 5 == 0:
        print(f"{n:,}개 chunk 처리 / 누적 접근성 row: {access_rows:,}")

grid_category_access = grid_access_series.reset_index(name="H3SFCA_접근성")
grid_category_access = grid_category_access.merge(
    row_count,
    on=["접근수단", "GRID_CD", "중분류"],
    how="left"
)
grid_category_access = grid_category_access.merge(
    grid_meta,
    on="GRID_CD",
    how="left"
)

print("접근성 계산 row:", f"{access_rows:,}")
print("격자-중분류 접근성 테이블:", grid_category_access.shape)
display(grid_category_access.head())

## 5. 결과 검토 및 저장

- 격자·접근수단·중분류별 H3SFCA 접근성 결과를 저장함.
- 격자 요약 테이블은 평균 접근성, 최저 접근성, 접근 가능한 중분류 수를 함께 제공함.
- 해석 시 중분류별 시설 수와 수요 규모가 다르므로, 절대값 비교보다 같은 중분류 내부의 상대적 취약지역 비교에 우선 활용함.

In [ ]:
category_summary = (
    grid_category_access
    .groupby(["접근수단", "중분류"], as_index=False)
    .agg(
        격자수=("GRID_CD", "nunique"),
        평균_H3SFCA=("H3SFCA_접근성", "mean"),
        중앙값_H3SFCA=("H3SFCA_접근성", "median"),
        하위10퍼센트=("H3SFCA_접근성", lambda x: x.quantile(0.1)),
        상위90퍼센트=("H3SFCA_접근성", lambda x: x.quantile(0.9)),
        평균_접근가능가맹점수=("접근가능_가맹점수", "mean")
    )
    .sort_values(["접근수단", "평균_H3SFCA"], ascending=[True, False])
)

grid_summary = (
    grid_category_access
    .groupby(["접근수단", "GRID_CD"], as_index=False)
    .agg(
        평균_H3SFCA=("H3SFCA_접근성", "mean"),
        최저_H3SFCA=("H3SFCA_접근성", "min"),
        접근가능_중분류수=("중분류", "nunique"),
        평균_접근가능가맹점수=("접근가능_가맹점수", "mean"),
        시군구_격자=("시군구_격자", "first"),
        행정동_격자=("행정동_격자", "first"),
        문화누리대상자_추정_인구수=("문화누리대상자_추정_인구수", "first")
    )
)

hjd_summary = (
    grid_category_access
    .groupby(["접근수단", "시군구_격자", "행정동_격자", "중분류"], as_index=False)
    .agg(
        평균_H3SFCA=("H3SFCA_접근성", "mean"),
        중앙값_H3SFCA=("H3SFCA_접근성", "median"),
        격자수=("GRID_CD", "nunique"),
        평균_접근가능가맹점수=("접근가능_가맹점수", "mean")
    )
)

print("중분류별 요약")
display(category_summary)

print("격자 요약")
display(grid_summary.head())

print("행정동-중분류 요약")
display(hjd_summary.head())

grid_category_access.to_csv(
    ACCESS_OUTPUT_PATH / "h3sfca_격자_중분류_접근성.csv",
    index=False,
    encoding="utf-8-sig"
)
facility_ratio.to_csv(
    ACCESS_OUTPUT_PATH / "h3sfca_가맹점_공급수요비.csv",
    index=False,
    encoding="utf-8-sig"
)
grid_summary.to_csv(
    ACCESS_OUTPUT_PATH / "h3sfca_격자_요약.csv",
    index=False,
    encoding="utf-8-sig"
)
hjd_summary.to_csv(
    ACCESS_OUTPUT_PATH / "h3sfca_행정동_중분류_요약.csv",
    index=False,
    encoding="utf-8-sig"
)
category_summary.to_csv(
    ACCESS_OUTPUT_PATH / "h3sfca_중분류_요약.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 폴더:", ACCESS_OUTPUT_PATH)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = category_summary.copy()
plot_df["label"] = plot_df["접근수단"] + " | " + plot_df["중분류"]
plot_df = plot_df.sort_values("평균_H3SFCA", ascending=True)

ax.barh(plot_df["label"], plot_df["평균_H3SFCA"], color="#e97855", alpha=0.85)
ax.set_title("접근수단·중분류별 평균 H3SFCA 접근성")
ax.set_xlabel("평균 H3SFCA 접근성")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
fig.savefig(ACCESS_IMAGE_PATH / "h3sfca_중분류별_평균접근성.png", dpi=200)
plt.show()

print("이미지 저장 폴더:", ACCESS_IMAGE_PATH)

## 실행 결과 요약

- 입력 접근성 pair 11,612,916건을 사용함.
- 접근수단·격자·중분류별 거리감쇠 합계 304,509건을 생성함.
- 가맹점별 공급-수요비 4,282건을 생성함.
- 최종 격자 요약 테이블 94,930건을 생성함: 대중교통 56,100건, 도보 38,830건.
- 도보는 도서·문화체험·음악·영상·체육시설·체육용품, 대중교통은 공연·관광지·미술·스포츠관람 분류가 포함됨.
- 산출물은 `notebooks/access/OUTPUT/h3sfca`, 이미지는 `notebooks/access/IMAGE/h3sfca`에 저장함.